<a href="https://colab.research.google.com/github/martatolos/eae-dsaa/blob/main/mlops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLOps: Full Lifecycle with MLflow

> **Goal:** Experience the complete MLOps lifecycle — data extraction, preprocessing, model training with experiment tracking, model registry, inference, drift detection, and REST serving — using a real Spotify tracks dataset and MLflow.

> **Requirements:** A free [ngrok account](https://dashboard.ngrok.com/signup) for the MLflow UI. All other dependencies are installed in the first cell.

> **Important:** Run cells top-to-bottom in a single Colab session. Variables set in earlier sections are used by later ones.

## Section 0: Environment Setup

### 0.1 Install Dependencies

Run this cell first. It installs all packages needed for this lab.

In [ ]:
%pip install mlflow==2.22.0 scikit-learn==1.8.0 polars==1.30.0 evidently==0.7.5 pyngrok==7.2.2 numpy==2.4.4 pandas==3.0.2 flask==3.1.0 requests==2.32.3 matplotlib --quiet

### 0.2 Clone Repo and Set Working Directory

We clone the repository so we have access to the bundled Spotify dataset in `data/`.

In [ ]:
import os
from pathlib import Path

!git clone https://github.com/martatolos/eae-dsaa.git /content/eae-dsaa --quiet

BASE_DIR = Path("/content/eae-dsaa")
DATA_DIR = BASE_DIR / "data"
os.chdir(BASE_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Dataset exists: {(DATA_DIR / 'spotify_tracks.csv').exists()}")

### 0.3 Start MLflow Server

MLflow needs a **SQLite backend** (not just a file store) to support the Model Registry feature we use in Section 5. We start it as a background process and poll until it responds.

> **Why SQLite?** MLflow's Model Registry (versioning, aliases) requires a database. SQLite is built into Python — no extra installation needed.

In [ ]:
import subprocess
import time
import requests as req

mlflow_process = subprocess.Popen(
    ["mlflow", "server",
     "--backend-store-uri", "sqlite:///mlflow.db",
     "--default-artifact-root", str(BASE_DIR / "mlruns"),
     "--host", "0.0.0.0",
     "--port", "5000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Poll until server responds (MLflow root / returns 200 when ready)
for i in range(20):
    try:
        r = req.get("http://localhost:5000/", timeout=2)
        if r.status_code == 200:
            print(f"MLflow server ready after {(i+1)*2}s")
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError("MLflow server did not start in time — check the process")

### 0.4 Expose MLflow UI via ngrok

**You need a free ngrok account for this step.**

1. Go to https://dashboard.ngrok.com/signup and create a free account
2. Copy your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
3. Paste it in the cell below

This gives you a public URL to access the MLflow UI from your browser.

In [ ]:
from pyngrok import ngrok

# Paste your ngrok authtoken here:
NGROK_TOKEN = "YOUR_TOKEN_HERE"

ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(5000)
MLFLOW_PUBLIC_URL = tunnel.public_url  # extract clean URL string
print(f"MLflow UI: {MLFLOW_PUBLIC_URL}")
print("Open this URL in a new tab — you should see an empty MLflow Experiments page.")

### 0.5 Configure MLflow Tracking URI

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
# Verify: list experiments (returns empty list on a fresh server)
client = mlflow.MlflowClient()
experiments = client.search_experiments()
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiments: {len(experiments)} (expected 0 on a fresh server)")

## Section 1: Data Extraction

### Why Data Extraction is a Distinct Step

In production ML pipelines, data extraction is always separated from preprocessing:
- **Auditability**: you can always reprocess from the raw source
- **Reproducibility**: the raw file is immutable; every transform is tracked
- **Data lake pattern**: raw data lands in one place, processed data in another

Here we load from our bundled CSV (committed to the repo), mimicking a read from a data lake.

**Polars `scan_csv` vs `read_csv`:** `scan_csv()` returns a `LazyFrame` — data is NOT loaded into memory yet. Operations you chain on it are recorded as a query plan. Only `.collect()` executes the plan. For our 5K-row dataset the difference is small, but the pattern scales well.

In [ ]:
import polars as pl

# scan_csv = lazy (returns LazyFrame, no data loaded yet)
raw_lf = pl.scan_csv(DATA_DIR / "spotify_tracks.csv")

# collect() executes the query plan and materialises the DataFrame
raw_df = raw_lf.collect()

print(f"Shape: {raw_df.shape}")
print(f"\nSchema:")
print(raw_df.schema)
print(f"\nFirst 5 rows:")
raw_df.head()

In [ ]:
# Save raw data (simulating writing to a data lake landing zone)
raw_df.write_csv(DATA_DIR / "raw_data.csv")
print(f"Saved raw_data.csv ({raw_df.shape[0]} rows)")

## Section 1.5: Exploratory Data Analysis

### Why EDA Before Preprocessing

Before writing a single transform, we need to understand:
- Is `popularity` (our target) normally distributed or skewed?
- Which features correlate with popularity?
- Are features on very different scales? (Hint: yes — `loudness` is in dB, `duration_ms` is in milliseconds)
- How balanced is the genre distribution? (Dataset has 6 genres, balanced at ~900 tracks each)

These observations directly inform our preprocessing choices.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Work in pandas for visualization convenience
eda_df = raw_df.to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Target distribution
axes[0].hist(eda_df["popularity"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Popularity Distribution")
axes[0].set_xlabel("Popularity (0-100)")
axes[0].set_ylabel("Track Count")

# Genre distribution
genre_counts = eda_df["genre"].value_counts()
axes[1].barh(genre_counts.index, genre_counts.values, color="steelblue")
axes[1].set_title("Tracks per Genre")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Feature correlations with popularity (bar chart)
numeric_cols = ["danceability", "energy", "loudness", "speechiness",
                "acousticness", "instrumentalness", "liveness", "valence",
                "tempo", "duration_ms", "explicit", "year"]

corr = eda_df[numeric_cols + ["popularity"]].corr()["popularity"].drop("popularity").sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["salmon" if v < 0 else "steelblue" for v in corr.values]
ax.barh(corr.index, corr.values, color=colors)
ax.set_title("Feature Correlation with Popularity")
ax.set_xlabel("Pearson Correlation")
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

top_positive = corr.tail(2).index.tolist()
print(f"Top positive correlations: {top_positive}")

In [ ]:
# Scatter plots: top 2 correlated features vs popularity
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, feat in enumerate(top_positive):
    axes[i].scatter(eda_df[feat], eda_df["popularity"], alpha=0.2, s=5, color="steelblue")
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel("popularity")
    axes[i].set_title(f"{feat} vs popularity")
plt.tight_layout()
plt.show()

In [ ]:
# Feature scale comparison — this is why we normalize
print("Feature ranges (note the very different scales):")
print(eda_df[["loudness", "duration_ms", "tempo", "danceability"]].describe().round(2))
print()
print(eda_df.describe().round(2))

## Section 2: Preprocessing

### Preprocessing Pipeline

Preprocessing transforms raw data into model-ready features. The same transformations **must** be applied consistently at both training and inference time — inconsistency here is one of the most common bugs in production ML.

We use Polars' lazy API to chain all transforms, then `.collect()` once to execute.

**One-hot vs ordinal encoding for genre:** Genre is a **nominal** categorical variable — there is no inherent order between pop, rock, and hip-hop. We use one-hot encoding (one binary column per genre, drop-first) rather than ordinal encoding (which would imply a false numerical ordering). Tree-based models handle many binary features well.

In [ ]:
import polars as pl

# Reload raw data as a LazyFrame
raw_lf = pl.scan_csv(DATA_DIR / "raw_data.csv")

# Step 1: Encode binary column
# Step 2: Normalise loudness and duration_ms to [0, 1]
# Compute min/max on the full dataset (in production, fit on train set only)
raw_collected = raw_lf.collect()
LOUDNESS_MIN = float(raw_collected["loudness"].min())
LOUDNESS_MAX = float(raw_collected["loudness"].max())
DURATION_MIN = float(raw_collected["duration_ms"].min())
DURATION_MAX = float(raw_collected["duration_ms"].max())

preprocessed_lf = raw_lf.with_columns([
    pl.col("explicit").cast(pl.Int8),
    ((pl.col("loudness") - LOUDNESS_MIN) / (LOUDNESS_MAX - LOUDNESS_MIN)).alias("loudness"),
    ((pl.col("duration_ms") - DURATION_MIN) / (DURATION_MAX - DURATION_MIN)).alias("duration_ms"),
])

print(f"Normalisation params saved for inference:")
print(f"  loudness : [{LOUDNESS_MIN:.2f}, {LOUDNESS_MAX:.2f}]")
print(f"  duration_ms: [{DURATION_MIN:.0f}, {DURATION_MAX:.0f}]")

In [ ]:
import pandas as pd

# Step 3: One-hot encode genre (via pandas get_dummies, then back to Polars)
preprocessed_df = preprocessed_lf.collect().to_pandas()
genre_dummies = pd.get_dummies(preprocessed_df["genre"], prefix="genre", drop_first=True, dtype=int)
GENRE_COLUMNS = list(genre_dummies.columns)

preprocessed_df = pd.concat([preprocessed_df.drop(columns=["genre"]), genre_dummies], axis=1)

# Convert back to Polars
preprocessed_df_pl = pl.from_pandas(preprocessed_df)

print(f"Genre columns created ({len(GENRE_COLUMNS)}): {GENRE_COLUMNS}")
print(f"\nProcessed shape: {preprocessed_df_pl.shape}")

In [ ]:
# Save processed data
preprocessed_df_pl.write_csv(DATA_DIR / "processed_data.csv")
print(f"Saved processed_data.csv {preprocessed_df_pl.shape}")

preprocessed_df_pl.head(3)

**Preprocessing artifacts stored as notebook variables for use in later sections:**
- `LOUDNESS_MIN`, `LOUDNESS_MAX` — for normalising new loudness values at inference time
- `DURATION_MIN`, `DURATION_MAX` — for normalising new duration values at inference time
- `GENRE_COLUMNS` — the ordered list of one-hot column names (must match exactly at inference)
- `preprocessed_df` — the pandas DataFrame used as Evidently reference in Section 6

## Section 3: Data Preparation

### Config-Driven Feature Selection

In production, feature lists and split parameters live in config files — not hardcoded in training scripts. This means swapping features is a config change, not a code change.

> **Note:** `GENRE_COLUMNS` was set in Section 2. This section must run in the same kernel session as Section 2.

In [ ]:
# CONFIG uses GENRE_COLUMNS from Section 2 (must run after Section 2)
CONFIG = {
    "features": [
        "danceability", "energy", "loudness", "speechiness",
        "acousticness", "instrumentalness", "liveness", "valence",
        "tempo", "duration_ms", "explicit", "year",
        *GENRE_COLUMNS,  # one-hot genre columns from Section 2
    ],
    "target": "popularity",
    "test_size": 0.2,
    "random_state": 42,
}

print(f"Features ({len(CONFIG['features'])}):")
for f in CONFIG["features"]:
    print(f"  {f}")
print(f"\nTarget: {CONFIG['target']}")

### Train/Test Split

We convert to pandas here — scikit-learn requires pandas DataFrames or numpy arrays, not Polars. The conversion happens at this **boundary** between our data pipeline (Polars) and our ML pipeline (sklearn).

In [ ]:
import pandas as pd
import polars as pl
from sklearn.model_selection import train_test_split

# Reload processed data and convert to pandas at the sklearn boundary
processed_df = pl.read_csv(DATA_DIR / "processed_data.csv").to_pandas()

X = processed_df[CONFIG["features"]]
y = processed_df[CONFIG["target"]]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["random_state"],
)

print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")
print(f"Features: {X_train.shape[1]}")
print(f"\nData leakage check: test set is held out from all tuning and training decisions.")

In [ ]:
# Save splits for recovery if kernel restarts
X_train.to_csv(DATA_DIR / "x_train.csv", index=False)
X_test.to_csv(DATA_DIR / "x_test.csv", index=False)
y_train.to_csv(DATA_DIR / "y_train.csv", index=False)
y_test.to_csv(DATA_DIR / "y_test.csv", index=False)
print("Saved: x_train.csv, x_test.csv, y_train.csv, y_test.csv")

## Section 4: Model Training + Experiment Tracking

### Experiment Tracking with MLflow

Without tracking, ML experiments are invisible: "which model did I train? what params? what was the accuracy?" MLflow solves this by logging everything — params, metrics, data snapshots, and the model artifact — to a persistent store you can query and compare.

All three models run in a **single experiment** (`spotify-popularity`) so you can compare them side-by-side in the UI. Grids are intentionally smaller than production for classroom timing (~5 minutes total on Colab free tier).

In [ ]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mlflow.set_experiment("spotify-popularity")

MODELS = {
    "LinearRegression": {
        "class": LinearRegression,
        "params": {"fit_intercept": [True, False]},
        "n_jobs": None,
    },
    "RandomForestRegressor": {
        "class": RandomForestRegressor,
        "params": {
            "n_estimators": [50, 100],
            "max_depth": [5, 10],
            "min_samples_split": [2, 5],
        },
        "n_jobs": -1,
    },
    "GradientBoostingRegressor": {
        "class": GradientBoostingRegressor,
        "params": {
            "n_estimators": [100, 200],
            "learning_rate": [0.05, 0.1],
            "max_depth": [3, 5],
        },
        "n_jobs": None,  # GB is sequential — n_jobs has no effect
    },
}

print("Models to train:", list(MODELS.keys()))
print("\nGrid sizes (intentionally reduced from production for class timing):")
for name, cfg in MODELS.items():
    n = 1
    for v in cfg["params"].values():
        n *= len(v)
    print(f"  {name}: {n} combos x 5 folds = {n*5} fits")

In [ ]:
results = []
best_run_id = None
best_r2 = -np.inf
best_model_name = None

for model_name, model_cfg in MODELS.items():
    print(f"\nTraining {model_name}...")

    estimator = model_cfg["class"]()
    grid_search = GridSearchCV(
        estimator,
        model_cfg["params"],
        cv=5,
        scoring="neg_mean_squared_error",
        n_jobs=model_cfg["n_jobs"],
        verbose=0,
    )

    with mlflow.start_run(run_name=model_name) as run:
        grid_search.fit(X_train, y_train)
        best_estimator = grid_search.best_estimator_

        y_pred = best_estimator.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        mlflow.log_params(grid_search.best_params_)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

        # Log grid search results CSV as artifact
        cv_path = f"/tmp/{model_name}_cv_results.csv"
        pd.DataFrame(grid_search.cv_results_).to_csv(cv_path, index=False)
        mlflow.log_artifact(cv_path, "grid_search")

        # Log model artifact only (NOT registered here — registration happens in Section 5)
        input_example = X_train.head(5)
        signature = mlflow.models.infer_signature(X_train, y_pred)
        mlflow.sklearn.log_model(
            best_estimator,
            artifact_path="model",
            signature=signature,
            input_example=input_example,
        )

        run_id = run.info.run_id

    results.append({"model": model_name, "rmse": rmse, "mae": mae, "r2": r2, "run_id": run_id})

    if r2 > best_r2:
        best_r2 = r2
        best_run_id = run_id
        best_model_name = model_name

    print(f"  RMSE={rmse:.3f}  MAE={mae:.3f}  R2={r2:.3f}")

print(f"\nBest model: {best_model_name} (R2={best_r2:.3f})")
print(f"Best run ID: {best_run_id}")

In [ ]:
# Comparison table
results_df = pd.DataFrame(results).set_index("model").drop(columns=["run_id"])
print("\nModel Comparison (test set):")
print(results_df.round(4).to_string())
print("\nNext: open the MLflow UI, click 'spotify-popularity', select all 3 runs, and click Compare.")

### What to do in the MLflow UI

1. Open the URL printed in Section 0.4
2. Click the **spotify-popularity** experiment in the left sidebar
3. Select all 3 runs and click **Compare**
4. Check the **Metrics** tab — which model has the lowest RMSE and highest R2?
5. Check the **Parameters** tab to see the best hyperparameters per model
6. Click any run to see its artifacts (model files, grid search CSV)

## Section 5: Model Registry

In [ ]:
# PLACEHOLDER_SECTION_5

## Section 6: Inference + Data Drift Detection

In [ ]:
# PLACEHOLDER_SECTION_6

## Section 7: Model Serving (REST API)

In [ ]:
# PLACEHOLDER_SECTION_7